# Instance creation stats

This script generate a summary of how many Instance records were created by each staff user from a user-supplied number of days in the past until today.

## 1. Environment setup


In [ ]:
# !pip install pandas requests

import pandas as pd
import requests
from datetime import datetime, timedelta   

pd.set_option('display.max_columns', None)


## 2. Login
### Note: This script references patron data, so the login credentials must be able to access user accounts in FOLIO. 


In [ ]:

%run folio_auth.ipynb

## 3. A small helper for paginated GET requests

FOLIO endpoints typically page results via `limit`/`offset` query params and return a
JSON object with a records array plus a total count. This helper loops until it has
everything. **Confirm the pagination param names and the response envelope key names
against your instance** — I'm using common FOLIO conventions here, but I have not
verified them against current docs.


In [ ]:
def fetch_all_records(endpoint, records_key, limit=1000):
    """
    Fetch all records from a paginated FOLIO endpoint.

    endpoint: path like "/accounts", "/groups", "/users"
    records_key: the JSON key holding the list of records, e.g. "accounts", "usergroups", "users"
    """
    all_records = []
    offset = 0
    base_url = OKAPI_URL.copy()
    headers = HEADERS.copy()


    while True:
        response = requests.get(
            f"{base_url}{endpoint}",
            headers=headers,
            params={"limit": limit, "offset": offset},
        )
        response.raise_for_status()  # fail loudly and clearly if something's wrong
        payload = response.json()

        batch = payload.get(records_key, [])
        all_records.extend(batch)

        if len(batch) < limit:
            break  # last page
        offset += limit

    return all_records

## 4. Get # of Days to Search from Input
The number entered will be used to search for Instance records with a date created of {number of days} - today's date. You can choose to bypass this and use a standard calculation if you prefer not to have an input.


In [ ]:

while True:
    try:
        from_days = int(input("Enter the number of days to search back from today, or press Enter to use the default of 90 days: ") or 90)
        if from_days < 0:  
            print("Number cannot be negative")
            continue
        else:                
            date_x_days_ago = datetime.now() - timedelta(days=from_days)
            search_date =str(date_x_days_ago.strftime("%Y-%m-%d"))
            print("You entered:", from_days, "days. The search will look for Instance records created since: ", search_date)
            break
    except ValueError:
        print("Please enter a valid number.")



## 4. Pull data from each endpoint

TODO (FOLIO-specific): confirm the `records_key` for each — FOLIO's convention is
usually the plural of the resource, but it varies (e.g. `/groups` often returns
`"usergroups"` rather than `"groups"` — **check this**, I'm not certain of the exact
key for your instance).


In [ ]:

def fetch_all_records(endpoint, records_key, limit=100, query=None):
    all_records = []
    offset = 0
    base_url = OKAPI_URL
    headers = HEADERS.copy() if 'HEADERS' in globals() else {"X-Okapi-Tenant": TENANT, "Content-Type": "application/json"}
    if 'token' in globals():
        headers["Authorization"] = f"Bearer {token}"

    while True:
        params = {"limit": limit, "offset": offset}
        if query:
            params["query"] = query
        response = session.get(f"{base_url}{endpoint}", headers=headers, params=params)
        response.raise_for_status()
        payload = response.json()

        batch = payload.get(records_key, [])
        all_records.extend(batch)

        if len(batch) < limit:
            break
        offset += limit

    return all_records


staff_raw = fetch_all_records(
    "/users",
    records_key="users",
    query='type=="staff"',
) 


# instances_raw = fetch_all_records(
#     "/inventory/instances",
#     records_key="instances",
#     query='metadata.createdDate >= ' + search_date,
# )

instances_raw = fetch_all_records(
    "/instance-storage/instances",
    records_key="instances",
    query='metadata.createdDate >= ' + search_date,
)

print(f"staff:     {len(staff_raw)}")
print(f"instances: {len(instances_raw)}")


In [ ]:
# Inspect staff records
staff_df    = pd.DataFrame(staff_raw)
staff_df.head()

In [ ]:
#Inspect instance records
instances_df   = pd.DataFrame(instances_raw)
instances_df.head()

## 5. Inspect join keys before merging

This is the step worth slowing down for. Before merging, check:
- Do the key columns actually exist under the names you expect?
- Are the data types consistent between the two sides of each join (e.g. both strings)?
- Any leading/trailing whitespace or case differences?


In [ ]:
print(staff_df.columns.tolist())
print(instances_df.columns.tolist())

In [ ]:
# Spot-check types of the columns you intend to join on
# Since the user ID in the instance records is nested in the metadata dictionary, we need to extract it first
print(staff_df['id'].dtype)
print(instances_df['metadata'].apply(lambda x: x.get('createdByUserId') if isinstance(x, dict) else None).dtype)

## 6. Merge

Two joins: accounts → users, then that result → groups.

Starting with `how='left'` keeps every account row even if a match isn't found, so you
can see what didn't match rather than silently losing rows.


In [ ]:
instances_users = instances_df.merge(staff_df, left_on=instances_df['metadata'].apply(lambda x: x.get('createdByUserId') if isinstance(x, dict) else None), right_on='id', how='left', suffixes=('_instance', '_user')
)
instances_users.head()

## 7. Validate the merge

Common beginner pitfall: a one-to-many relationship silently multiplying rows, or a
type mismatch causing everything to come back unmatched. Check both.


In [ ]:
print("Original Instance rows:", len(instances_df))
print("After merging with staff:  ", len(instances_users))

## 8. Analyze the combined dataset

Now that accounts, users, and groups are joined, you can ask questions that span all
three — e.g. total fee/fine amounts by patron group. Adjust field names to match your
actual `/accounts` schema (commonly something like `amount` or `remaining`).


In [ ]:
summary =instances_users.groupby('username')['id_instance'].count().sort_values(ascending=False)
print("Summary of instance records created by each staff user from", search_date, "to today:")
print(summary)